<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day08-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 8, Part 2 discussion — Where does homology leakage come from?

The book page clustered the 700 localization proteins at **30%** identity
and found that 31.3% of validation proteins had a homolog in the training
set, and that those were classified correctly far more often (0.675 vs.
0.509).

Would a stricter or looser cutoff change the picture? Here the same 700
proteins have been clustered with MMseqs2 at **30%, 50%, 70% and 90%**
identity (precomputed, so this runs on Colab without MMseqs2).

**Predict first:** as the cutoff rises from 30% to 90%, what happens to
(i) the fraction of validation proteins with a "homolog" in training and
(ii) the accuracy advantage those proteins get?

In [1]:
import os, re
import numpy as np
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

UNIPROT_CLASSES = {
    "Cytoplasm": "SL-0086",
    "Nucleus": "SL-0191",
    "Mitochondrion": "SL-0173",
    "Secreted": "SL-0243",
    "Cell membrane": "SL-0039",
}
CLASS_NAMES = list(UNIPROT_CLASSES.keys())

def fetch_uniprot(sl_code, size=500):
    query = f"organism_id:9606 AND reviewed:true AND cc_scl_term:{sl_code} AND length:[50 TO 500]"
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/search",
        params={"query": query, "fields": "accession,sequence,cc_subcellular_location",
                "format": "tsv", "size": size},
        timeout=60,
    )
    r.raise_for_status()
    rows = []
    for line in r.text.strip().split("\n")[1:]:
        parts = line.split("\t")
        if len(parts) == 3:
            rows.append(tuple(parts))  # (accession, sequence, location_text)
    return rows

def is_unambiguous_single_location(location_text, target):
    '''Keep only entries whose SUBCELLULAR LOCATION comment names exactly
    one location term (matching the target), ignoring free-text Notes,
    evidence-code tags, and skipping isoform-specific annotations.'''
    location_text = re.sub(r"Note=.*", "", location_text)
    location_text = re.sub(r"\{[^}]*\}", "", location_text)
    if "Isoform" in location_text:
        return False
    terms = set()
    for statement in [s.strip() for s in location_text.split(".") if s.strip()]:
        body = statement.split(":", 1)[-1] if ":" in statement else statement
        top_term = re.split(r"[,;]", body)[0].strip()
        if top_term:
            terms.add(top_term)
    return len(terms) == 1 and next(iter(terms)) == target

entries_by_class = {}
for class_name, sl_code in UNIPROT_CLASSES.items():
    raw = fetch_uniprot(sl_code)
    clean = [(acc, seq) for acc, seq, loc in raw if is_unambiguous_single_location(loc, class_name)]
    entries_by_class[class_name] = clean
    print(f"{class_name:15s} raw hits: {len(raw):4d}   unambiguous single-location: {len(clean):4d}")

Cytoplasm       raw hits:  500   unambiguous single-location:  424


Nucleus         raw hits:  500   unambiguous single-location:  497


Mitochondrion   raw hits:  500   unambiguous single-location:  140


Secreted        raw hits:  500   unambiguous single-location:  488


Cell membrane   raw hits:  500   unambiguous single-location:  338


In [2]:
N_PER_CLASS = min(len(v) for v in entries_by_class.values())
rng = np.random.RandomState(0)
balanced_accs, balanced_seqs, balanced_labels = [], [], []
for class_idx, class_name in enumerate(CLASS_NAMES):
    pool = entries_by_class[class_name]
    for i in rng.choice(len(pool), N_PER_CLASS, replace=False):
        balanced_accs.append(pool[i][0]); balanced_seqs.append(pool[i][1]); balanced_labels.append(class_idx)

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
def one_hot_encode(sequence, length=150):
    encoded = np.zeros((length, 20), dtype=np.float32)
    for position, residue in enumerate(sequence[:length]):
        if residue in AA_TO_INDEX:
            encoded[position, AA_TO_INDEX[residue]] = 1.0
    return encoded.flatten()
X = np.stack([one_hot_encode(s) for s in balanced_seqs])
y = np.array(balanced_labels)

URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/day08-subcell-mmseqs-multi.tsv"
LOCAL = "data/day08-subcell-mmseqs-multi.tsv"
text = open(LOCAL).read() if os.path.exists(LOCAL) else requests.get(URL, timeout=30).text
header, *rows = [line.split("\t") for line in text.strip().split("\n")]
table = {r[0]: r[1:] for r in rows}
THRESHOLDS = [h.replace("cluster_", "") + "%" for h in header[1:]]
groups = {t: np.array([table.get(a, [a] * 4)[k] for a in balanced_accs]) for k, t in enumerate(THRESHOLDS)}
for t in THRESHOLDS:
    print(f"{t} identity: {len(set(groups[t]))} clusters for {len(y)} proteins")

30% identity: 551 clusters for 700 proteins
50% identity: 665 clusters for 700 proteins
70% identity: 694 clusters for 700 proteins
90% identity: 699 clusters for 700 proteins


The same ten random splits and logistic-regression baseline as the book
page. The model is fitted once per split, then each threshold only
changes which validation proteins count as "having a homolog in
training":

In [3]:
idx = np.arange(len(y))
runs = []
for seed in range(10):
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=seed)
    va, _ = train_test_split(tmp, test_size=0.5, stratify=y[tmp], random_state=seed)
    pred = LogisticRegression(max_iter=2000, random_state=0).fit(X[tr], y[tr]).predict(X[va])
    runs.append((tr, va, pred))

print(f"{'cutoff':>7s} {'leak':>7s} {'acc with homolog':>18s} {'n':>5s} {'acc without':>12s}")
for t in THRESHOLDS:
    g = groups[t]
    leak, hit, miss = [], [], []
    for tr, va, pred in runs:
        in_train = set(g[tr])
        has = np.array([g[i] in in_train for i in va])
        leak.append(has.mean())
        hit += list(pred[has] == y[va][has]); miss += list(pred[~has] == y[va][~has])
    print(f"{t:>7s} {np.mean(leak):7.1%} {np.mean(hit):18.3f} {len(hit):5d} {np.mean(miss):12.3f}")

 cutoff    leak   acc with homolog     n  acc without
    30%   31.3%              0.675   329        0.509
    50%    7.8%              0.537    82        0.563
    70%    0.9%              0.444     9        0.562
    90%    0.2%              1.000     2        0.560


**Discuss:**

1. At which cutoff does the "homolog in training" advantage disappear?
   What does that tell you about which homologs this one-hot linear model
   is exploiting: close relatives or distant ones?
2. Look at the `n` column. Why are the accuracies at 70% and 90% not
   worth interpreting?
3. Many published sequence-based predictors partition at 25-30%
   identity. Based on this table, would 50% have been enough *for this
   dataset*? Would you expect the same for a model that sees whole
   sequences, like the CNNs of Day 10 or the protein language models of
   Day 12?